In [1]:
import pandas as pd
import numpy as np
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

In [2]:
df_credit = pd.read_csv(r"C:/Users/senan/Downloads/credit_risk_dataset/credit_train.csv")
df_credit.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [3]:
print("Dataset Shape:", df_credit.shape)
print(df_credit.columns)
df_credit.info()

Dataset Shape: (614, 13)
Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 1

In [4]:
df_credit.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [5]:
# Fill missing values

for col in df_credit.columns:

    # For categorical columns
    if df_credit[col].dtype == "object":
        df_credit[col].fillna(df_credit[col].mode()[0], inplace=True)

    # For numerical columns
    else:
        df_credit[col].fillna(df_credit[col].median(), inplace=True)

In [6]:
df_credit.isnull().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [7]:
df_credit["Loan_Status"].value_counts()

Loan_Status
Y    422
N    192
Name: count, dtype: int64

In [8]:
df_credit = df_credit.drop_duplicates()

print("Duplicate Rows:", df_credit.duplicated().sum())

Duplicate Rows: 0


In [9]:
if "Loan_ID" in df_credit.columns:
    df_credit = df_credit.drop("Loan_ID", axis=1)

In [10]:
label_encoders = {}

for col in df_credit.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df_credit[col] = le.fit_transform(df_credit[col])
    label_encoders[col] = le

df_credit.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,1,0,0,0,0,5849,0.0,128.0,360.0,1.0,2,1
1,1,1,1,0,0,4583,1508.0,128.0,360.0,1.0,0,0
2,1,1,0,0,1,3000,0.0,66.0,360.0,1.0,2,1
3,1,1,0,1,0,2583,2358.0,120.0,360.0,1.0,2,1
4,1,0,0,0,0,6000,0.0,141.0,360.0,1.0,2,1


In [11]:
X_credit = df_credit.drop("Loan_Status", axis=1)
y_credit = df_credit["Loan_Status"]

print("Features:", X_credit.shape)
print("Target:", y_credit.shape)

Features: (614, 11)
Target: (614,)


In [12]:
X_credit_train, X_credit_test, y_credit_train, y_credit_test = train_test_split(
    X_credit,
    y_credit,
    test_size=0.2,
    random_state=42,
    stratify=y_credit
)

In [13]:
credit_scaler = StandardScaler()

X_credit_train_scaled = credit_scaler.fit_transform(X_credit_train)
X_credit_test_scaled = credit_scaler.transform(X_credit_test)

In [14]:
credit_log = LogisticRegression(max_iter=1000, random_state=42)

credit_log.fit(X_credit_train_scaled, y_credit_train)

y_pred_log = credit_log.predict(X_credit_test_scaled)

print("Logistic Regression")
print(classification_report(y_credit_test, y_pred_log))

Logistic Regression
              precision    recall  f1-score   support

           0       0.96      0.58      0.72        38
           1       0.84      0.99      0.91        85

    accuracy                           0.86       123
   macro avg       0.90      0.78      0.81       123
weighted avg       0.88      0.86      0.85       123



In [15]:
credit_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

credit_rf.fit(X_credit_train, y_credit_train)

y_pred_rf = credit_rf.predict(X_credit_test)

print("Random Forest")
print(classification_report(y_credit_test, y_pred_rf))

Random Forest
              precision    recall  f1-score   support

           0       0.77      0.63      0.70        38
           1       0.85      0.92      0.88        85

    accuracy                           0.83       123
   macro avg       0.81      0.77      0.79       123
weighted avg       0.83      0.83      0.82       123



In [16]:
credit_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    eval_metric="logloss",
    random_state=42
)

credit_xgb.fit(X_credit_train, y_credit_train)

y_pred_xgb = credit_xgb.predict(X_credit_test)

print("XGBoost")
print(classification_report(y_credit_test, y_pred_xgb))

XGBoost
              precision    recall  f1-score   support

           0       0.80      0.63      0.71        38
           1       0.85      0.93      0.89        85

    accuracy                           0.84       123
   macro avg       0.82      0.78      0.80       123
weighted avg       0.83      0.84      0.83       123



In [17]:
credit_results = []

models = {
    "Logistic Regression": (credit_log, X_credit_test_scaled),
    "Random Forest": (credit_rf, X_credit_test),
    "XGBoost": (credit_xgb, X_credit_test)
}

for name, (model, X_eval) in models.items():
    y_pred = model.predict(X_eval)

    credit_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_credit_test, y_pred),
        "Precision": precision_score(y_credit_test, y_pred),
        "Recall": recall_score(y_credit_test, y_pred),
        "F1 Score": f1_score(y_credit_test, y_pred)
    })

credit_results_df = pd.DataFrame(credit_results)
credit_results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.861789,0.840000,0.988235,0.908108
1,Random Forest,0.829268,0.847826,0.917647,0.881356
2,XGBoost,0.837398,0.849462,0.929412,0.887640


In [18]:
joblib.dump(
    credit_log,
    "models/credit_risk_model.pkl"
)

joblib.dump(
    credit_scaler,
    "models/credit_scaler.pkl"
)

joblib.dump(
    label_encoders,
    "models/credit_label_encoders.pkl"
)

joblib.dump(
    X_credit.columns.tolist(),
    "models/credit_feature_columns.pkl"
)

print("Credit Risk Model Saved Successfully")

Credit Risk Model Saved Successfully


In [19]:
df_credit.to_csv("dataset/credit_risk_processed.csv", index=False)

print("Cleaned credit risk dataset saved successfully")

Cleaned credit risk dataset saved successfully


In [20]:
credit_results_df.to_csv(
    "reports/credit_model_results.csv",
    index=False
)

In [21]:
X_credit.columns

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area'],
      dtype='object')

In [22]:
df_credit.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,1,0,0,0,0,5849,0.0,128.0,360.0,1.0,2,1
1,1,1,1,0,0,4583,1508.0,128.0,360.0,1.0,0,0
2,1,1,0,0,1,3000,0.0,66.0,360.0,1.0,2,1
3,1,1,0,1,0,2583,2358.0,120.0,360.0,1.0,2,1
4,1,0,0,0,0,6000,0.0,141.0,360.0,1.0,2,1
